# Combining Mantis and TiViT Embeddings

[TiViT](https://github.com/ExplainableML/TiViT) turns a time series into a grayscale image and extracts features with a *frozen* Vision Transformer pre-trained on images. Since it looks at time series through a completely different lens than a time series foundation model, its representations turn out to be complementary to ours: concatenating the two improves classification accuracy over either one alone.

This notebook shows how to do that concatenation. TiViT is not a dependency of `mantis-tsfm`, and we do not want to make it one, so below we reimplement the few lines that are specific to it and rely on a small vision backbone that is quick to download.

The only extra requirement is `transformers`, which loads the vision backbone:

```bash
pip install transformers
```

## Read data

demonstration data set from the UCR collection

In [1]:
import numpy as np
import torch
import torch.nn.functional as F

data = [np.load(f'../data/GestureMidAirD1/{variable}_{set_name}.npy')
        for variable in ['X', 'y'] for set_name in ['train', 'test']]

X_train, X_test, y_train, y_test = data

def resize(X):
    X_scaled = F.interpolate(torch.tensor(X, dtype=torch.float), size=512, mode='linear', align_corners=False)
    return X_scaled.numpy()

X_train, X_test = resize(X_train), resize(X_test)

device = 'cpu' # set device

print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (208, 1, 512)
X_test dims:  (130, 1, 512)


## Mantis embeddings

Nothing special here, this is the usual feature extraction:

In [2]:
from mantis.architecture import MantisV1
from mantis.trainer import MantisTrainer

network = MantisV1(device=device)
network = network.from_pretrained("paris-noah/Mantis-8M")
mantis = MantisTrainer(device=device, network=network)

Z_train_mantis = mantis.transform(X_train)
Z_test_mantis = mantis.transform(X_test)

print("Mantis embeddings: ", Z_train_mantis.shape)

Mantis embeddings:  (208, 256)


## TiViT embeddings

The part that is really specific to TiViT is how a time series becomes an image: the series is scaled robustly, cut into *overlapping* windows of length $\sqrt{\text{seq len}}$ that are stacked as the rows of an image, each image gets its contrast adjusted, and the result is resized to the resolution the backbone expects and repeated over three channels. Everything after that is a forward pass through a frozen ViT, mean-pooling the token representations of one intermediate layer.

The class below is a stripped-down version of the [official implementation](https://github.com/ExplainableML/TiViT) that keeps a single Hugging Face backbone. Note that it assumes univariate series, as the image is built from one channel.

In [3]:
import json
import math

from huggingface_hub import hf_hub_download
from transformers import AutoModel


class TiViT:
    """Minimal single-backbone TiViT feature extractor for univariate time series."""

    def __init__(self, model_name='facebook/dinov2-small', layer_idx=6, stride=0.1, device='cpu'):
        self.vit = AutoModel.from_pretrained(model_name).to(device).eval()
        # we build the images ourselves, so out of the whole image processor of the backbone
        # only its normalization constants are needed
        with open(hf_hub_download(model_name, 'preprocessor_config.json')) as file:
            preprocessing = json.load(file)
        self.mean = torch.tensor(preprocessing['image_mean'], device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(preprocessing['image_std'], device=device).view(1, 3, 1, 1)
        self.layer_idx = layer_idx
        self.stride = stride
        self.device = device

    def ts2image(self, x, image_size=224):
        """(n_samples, 1, seq_len) time series -> (n_samples, 3, image_size, image_size) images."""
        # robust scaling along the time axis
        x = x.transpose(1, 2)
        quantiles = torch.tensor([0.75, 0.25], device=x.device, dtype=x.dtype)
        q75, q25 = torch.quantile(x, quantiles, dim=1, keepdim=True)
        x = (x - x.median(1, keepdim=True)[0]) / ((q75 - q25) + 1e-5)
        x = x.transpose(1, 2)

        # stack overlapping windows as the rows of an image
        seq_len = x.shape[-1]
        patch_size = int(math.sqrt(seq_len))
        stride_len = max(int(patch_size * self.stride), 1)
        remainder = (seq_len - patch_size) % stride_len
        if remainder:
            x = F.pad(x, (stride_len - remainder, 0), mode='replicate')
        images = x.unfold(dimension=2, size=patch_size, step=stride_len)

        # adjust the contrast of every image, then match the input format of the backbone
        low = images.amin(dim=(-2, -1), keepdim=True)
        high = images.amax(dim=(-2, -1), keepdim=True)
        images = ((images - low) / (high - low + 1e-5)) ** 0.8
        images = F.interpolate(images, size=(image_size, image_size), mode='nearest')
        return images.repeat(1, 3, 1, 1)

    def transform(self, x, batch_size=32):
        embeddings = []
        for start in range(0, x.shape[0], batch_size):
            batch = torch.tensor(x[start:start + batch_size], dtype=torch.float, device=self.device)
            images = (self.ts2image(batch) - self.mean) / self.std
            with torch.no_grad():
                hidden_states = self.vit(pixel_values=images, output_hidden_states=True).hidden_states
            # mean over the token dimension, as in the original implementation
            embeddings.append(hidden_states[self.layer_idx].mean(dim=1).cpu().numpy())
        return np.concatenate(embeddings)

In [4]:
tivit = TiViT(model_name='facebook/dinov2-small', layer_idx=6, device=device)

Z_train_tivit = tivit.transform(X_train)
Z_test_tivit = tivit.transform(X_test)

print("TiViT embeddings: ", Z_train_tivit.shape)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

TiViT embeddings:  (208, 384)


## Combining the two

Concatenating along the feature axis is all it takes. The two embeddings come from unrelated models, though, so their features do not necessarily live on the same scale, and one block could end up dominating the other. We therefore also try a variant where each block is $\ell_2$-normalized before being concatenated, and we evaluate every feature set both with a random forest and with a linear classifier:

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def l2_normalize(Z):
    return Z / np.linalg.norm(Z, axis=-1, keepdims=True)


feature_sets = {
    'Mantis': (Z_train_mantis, Z_test_mantis),
    'TiViT': (Z_train_tivit, Z_test_tivit),
    'Mantis + TiViT': (
        np.concatenate([Z_train_mantis, Z_train_tivit], axis=1),
        np.concatenate([Z_test_mantis, Z_test_tivit], axis=1),
    ),
    'Mantis + TiViT (l2)': (
        np.concatenate([l2_normalize(Z_train_mantis), l2_normalize(Z_train_tivit)], axis=1),
        np.concatenate([l2_normalize(Z_test_mantis), l2_normalize(Z_test_tivit)], axis=1),
    ),
}

predictors = {
    'random forest': lambda: RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
    'log. regression': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=0)),
}

print(f"{'features':<22}" + ''.join(f"{name:>18}" for name in predictors))
for features, (Z_train, Z_test) in feature_sets.items():
    accuracies = []
    for init_predictor in predictors.values():
        predictor = init_predictor()
        predictor.fit(Z_train, y_train)
        accuracies.append(np.mean(y_test == predictor.predict(Z_test)))
    print(f"{features:<22}" + ''.join(f"{acc:>18.4f}" for acc in accuracies))

features                   random forest   log. regression


Mantis                            0.6692            0.7154


TiViT                             0.6385            0.6923


Mantis + TiViT                    0.6923            0.7692


Mantis + TiViT (l2)               0.7231            0.7615


The combination beats both of its parts with either classifier, which is the point of the exercise: even this small vision backbone sees something in the data that Mantis does not.

The $\ell_2$-normalization, on the other hand, is not a clear win. It helps the random forest noticeably but slightly hurts the linear classifier, so on this data set there is no reason to prefer it. This matches how the reported results were produced, i.e. without any normalization before concatenation. Worth keeping in mind as a knob to try rather than a rule, and keep in mind that a single small data set says very little either way.

## Reproducing our results

The backbone above was chosen so that this notebook stays small and runs on a CPU, so its accuracy is **not** the one we report. Our numbers, given in the [Mantis ICML paper](https://arxiv.org/abs/2502.15637), are obtained with substantially larger vision encoders:

| Backbone | Checkpoint | Layer |
|---|---|---|
| OpenCLIP ViT-H-14 | `laion/CLIP-ViT-H-14-laion2B-s32B-b79K` | 14 |
| OpenCLIP ConvNeXt-XXL | `laion/CLIP-convnext-xxlarge-laion2b-s34b-b82k-augreg` | 15 |

Both use the same image construction as above (overlapping windows of length $\sqrt{\text{seq len}}$, i.e. `patch_size='sqrt'` and `stride=0.1`) with mean aggregation. They are OpenCLIP models rather than plain Hugging Face ones, so they need the `open_clip_torch` package and a different way of walking the layers. Instead of reimplementing all of that here, use the [official TiViT implementation](https://github.com/ExplainableML/TiViT), which supports these backbones directly and is described in [Roschmann et al.](https://arxiv.org/abs/2506.08641). The concatenation step stays exactly the same as in the cell above.